In [ ]:
# Clone repo (thay <username> bằng GitHub username của bạn)
!git clone https://github.com/viCore12/Day22-Track3-DPO-Alignment-Lab
%cd Day22-Track3-DPO-Alignment-Lab

# Cài deps + convert .py → .ipynb
!pip install -q unsloth trl peft bitsandbytes datasets wandb huggingface_hub jupytext
!jupytext --to notebook notebooks/*.py

# Set secrets (thay bằng keys thật)
import os
os.environ["COMPUTE_TIER"] = "T4"
os.environ["WANDB_API_KEY"] = ""
os.environ["WANDB_PROJECT"] = "lab22-dpo"
os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["HF_TOKEN"] = ""
os.environ["HF_REPO"] = "Nhanvi282/lab22-dpo-vn"

# NB1 (~10 min)

In [ ]:
%run notebooks/01_sft_mini.ipynb

# NB2 (~2 min)

In [ ]:
%run notebooks/02_preference_data.ipynb

# NB3 (\~30 min) — có W&B + NB4 (\~15 min) — có cross-judge

In [ ]:
# --- Lượt 1 ---
import os
os.environ["DPO_BETA"] = "0.05"
%run notebooks/03_dpo_train.ipynb
%run notebooks/04_compare_and_eval.ipynb

In [ ]:
# --- Lượt 2 ---
os.environ["DPO_BETA"] = "0.1"
%run notebooks/03_dpo_train.ipynb
%run notebooks/04_compare_and_eval.ipynb

In [ ]:
# --- Lượt 3 ---
os.environ["DPO_BETA"] = "0.5"
%run notebooks/03_dpo_train.ipynb
%run notebooks/04_compare_and_eval.ipynb

# NB5 (~20 min) — có Q5+HF push

In [ ]:
# [ignoring loop detection]
!pip install -q "transformers==4.43.3" "peft==0.11.1"
print("Đã cài đặt xong bản ổn định!")
print("Đang ép buộc khởi động lại Kernel để nhận thư viện mới...")
import time; time.sleep(2)
import os; os._exit(00)  # Lệnh này làm kernel tự động restart

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc

ROOT = "/kaggle/working/Day22-Track3-DPO-Alignment-Lab"

print("1. Tải Base Model 16-bit (tránh lỗi 4-bit của Unsloth)...")
model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-3B", 
    torch_dtype=torch.float16, 
    device_map="cpu"  # Dùng CPU RAM để tránh OOM GPU
)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-3B")

print("2. Đắp SFT Adapter và gộp...")
model = PeftModel.from_pretrained(model, f"{ROOT}/adapters/sft-mini")
model = model.merge_and_unload()

print("3. Đắp DPO Adapter và gộp...")
model = PeftModel.from_pretrained(model, f"{ROOT}/adapters/dpo-b0.1")  # Trỏ đúng thư mục DPO của bạn
model = model.merge_and_unload()

print("4. Lưu model FP16 hoàn chỉnh...")
model.save_pretrained(f"{ROOT}/adapters/merged-fp16")
tokenizer.save_pretrained(f"{ROOT}/adapters/merged-fp16")

del model
gc.collect()
print("🎉 ĐÃ LƯU XONG MERGED MODEL!")

In [ ]:
%%bash
ROOT="/kaggle/working/Day22-Track3-DPO-Alignment-Lab"

# 1. Tải và build llama.cpp (công cụ xuất GGUF chuẩn nhất)
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
make -j4

# 2. Convert model vừa gộp sang GGUF chuẩn (FP16)
python convert_hf_to_gguf.py $ROOT/adapters/merged-fp16 \
    --outfile $ROOT/gguf/model-fp16.gguf \
    --outtype f16

# 3. Nén xuống Q4_K_M (chuẩn)
./llama-quantize $ROOT/gguf/model-fp16.gguf $ROOT/gguf/model-q4_k_m.gguf q4_k_m

# 4. Nén xuống Q5_K_M (Lấy Bonus +3 điểm)
./llama-quantize $ROOT/gguf/model-fp16.gguf $ROOT/gguf/model-q5_k_m.gguf q5_k_m

# Dọn dẹp file FP16 nặng để đỡ tốn dung lượng Kaggle
rm $ROOT/gguf/model-fp16.gguf

echo "🎉 HOÀN TẤT XUẤT GGUF! Kiểm tra thư mục gguf/"
ls -lh $ROOT/gguf/

In [ ]:
import os
from huggingface_hub import HfApi

# Hãy thay bằng Token và Repo thật của bạn
HF_TOKEN = ""  
HF_REPO = "Nhanvi282/Qwen2.5-3B-DPO-Lab22" 

print(f"Đang đẩy file GGUF lên repo: {HF_REPO} ...")
api = HfApi()

# 1. Tạo Repo (nếu chưa có)
api.create_repo(
    repo_id=HF_REPO,
    private=False,
    token=HF_TOKEN,
    exist_ok=True
)

# 2. Đẩy toàn bộ thư mục gguf lên HF
api.upload_folder(
    folder_path="/kaggle/working/Day22-Track3-DPO-Alignment-Lab/gguf",
    repo_id=HF_REPO,
    repo_type="model",
    token=HF_TOKEN
)

print(f"🎉 Hoàn tất! Model của bạn đã lên mây: https://huggingface.co/{HF_REPO}")
print("Hãy copy link này dán vào phần Bonus trong file REFLECTION.md nhé!")

# NB6 (~45 min)

In [ ]:
# Cài đặt công cụ benchmark chuẩn của EleutherAI
!pip install -q git+https://github.com/EleutherAI/lm-evaluation-harness.git
!pip install -q vllm  # vLLM giúp chạy benchmark nhanh gấp 10 lần (tuỳ chọn)

print("✓ Đã cài đặt xong lm_eval!")

In [ ]:
import os
from pathlib import Path

# 1. Chuyển về đúng thư mục làm việc
%cd /kaggle/working/Day22-Track3-DPO-Alignment-Lab

# 2. Tạo đường tắt dpo -> dpo-b0.1
!ln -sf /kaggle/working/Day22-Track3-DPO-Alignment-Lab/adapters/dpo-b0.1 \
         /kaggle/working/Day22-Track3-DPO-Alignment-Lab/adapters/dpo

# 3. Kiểm tra và chạy NB6
if Path("adapters/dpo").exists() and Path("adapters/sft-mini").exists():
    print("✓ Các thư mục adapter đã sẵn sàng. Đang khởi động NB6...")
    %run notebooks/06_benchmark.ipynb
else:
    print("❌ Lỗi: Thiếu thư mục adapters/dpo hoặc adapters/sft-mini. Hãy kiểm tra lại.")